# Phase 2 - 01: 环境验证与模型推理

## 本节目标

1. 下载 Qwen3.5-2B 模型，验证 MPS 推理正常
2. 理解 Qwen3 的工具调用格式
3. 手动构造一个带工具调用的对话，为 GRPO 训练做准备

## 为什么用 Qwen3.5-2B？

- 比 Qwen2.5-1.5B 更新一代（2026年3月发布），能力更强
- M4 16G 可运行，M1 Ultra 更快
- 原生支持 tool use，有内置的工具调用格式

## 1. 验证环境

In [2]:
import torch
import transformers
import trl
import datasets

print(f"torch:          {torch.__version__}")
print(f"MPS available:  {torch.backends.mps.is_available()}")
print(f"transformers:   {transformers.__version__}")
print(f"trl:            {trl.__version__}")
print(f"datasets:       {datasets.__version__}")

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"\n使用设备: {DEVICE}")

/opt/homebrew/Caskroom/miniconda/base/envs/rl_study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch:          2.10.0
MPS available:  True
transformers:   5.3.0
trl:            0.29.1
datasets:       4.8.3

使用设备: mps


## 2. 下载并加载 Qwen3.5-2B

首次运行会从 HuggingFace 下载约 4GB 模型文件，下载完成后会缓存在 `~/.cache/huggingface/`。

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

# ModelScope 下载路径
MODEL_ID = os.path.expanduser("~/.cache/modelscope/hub/models/Qwen/Qwen3___5-2B")

print(f"加载 tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
print("tokenizer 加载完成")
print(f"vocab size: {tokenizer.vocab_size}")
print(f"chat template: {'有' if tokenizer.chat_template else '无'}")

加载 tokenizer: /Users/liqiang/.cache/modelscope/hub/models/Qwen/Qwen3___5-2B
tokenizer 加载完成
vocab size: 248044
chat template: 有


In [4]:
print(f"加载模型...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map=DEVICE,
    local_files_only=True,
)
model.eval()

param_count = sum(p.numel() for p in model.parameters()) / 1e9
print(f"模型加载完成，参数量: {param_count:.2f}B")
print(f"模型设备: {next(model.parameters()).device}")

`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


加载模型...


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 320/320 [00:04<00:00, 78.56it/s]


模型加载完成，参数量: 1.88B
模型设备: mps:0


## 3. 基础推理测试

先测试普通对话，确认模型推理正常。

In [5]:
def chat(messages, max_new_tokens=256, temperature=0.7):
    """简单封装：messages -> 生成文本"""
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # 只返回新生成的部分
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


# 测试基础对话
response = chat([
    {"role": "user", "content": "1 + 1 等于多少？"}
])
print("模型回答:", response)

模型回答: 1 + 1 等于 **2**。



## 4. 工具调用格式实验

### 背景：为什么需要统一格式？

GRPO 训练的核心思路：
- 给模型一道数学题
- 期望模型输出：`<think>推理过程</think><tool_call>计算表达式</tool_call><answer>最终答案</answer>`
- Reward 函数检查：格式是否正确？调用工具后答案是否正确？

我们先看看模型在没有训练的情况下，能否自然地调用工具。

In [6]:
# 系统提示：定义工具调用格式
SYSTEM_PROMPT = """你是一个数学助手，可以调用计算器工具解决计算问题。

请按照以下格式回答：
<think>
分析问题的思路
</think>
<tool_call>
Python 表达式（例如：2 + 3 * 4）
</tool_call>
<answer>
最终答案（数字）
</answer>
"""

# 测试一道简单数学题
question = "小明有 15 个苹果，送给同学 7 个，又买了 3 个，现在有多少个？"

response = chat([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": question},
], max_new_tokens=512)

print("问题:", question)
print("\n模型输出:")
print(response)

问题: 小明有 15 个苹果，送给同学 7 个，又买了 3 个，现在有多少个？

模型输出:
<tool_call>
Python 表达式（例如：15 - 7 + 3）
</tool_call>
<answer>
11
</answer>



## 5. 计算器工具实现

我们的「工具」就是 Python `eval`，安全地执行数学表达式。

In [7]:
import re
import math

# 允许的安全函数
SAFE_MATH_ENV = {
    "__builtins__": {},
    "abs": abs, "round": round, "int": int, "float": float,
    "sqrt": math.sqrt, "pow": math.pow, "floor": math.floor, "ceil": math.ceil,
}


def calculator(expression: str) -> str:
    """安全计算数学表达式，返回结果字符串"""
    try:
        # 清理表达式
        expr = expression.strip().replace("^", "**")
        result = eval(expr, SAFE_MATH_ENV)
        # 整数结果去掉小数点
        if isinstance(result, float) and result.is_integer():
            return str(int(result))
        return str(result)
    except Exception as e:
        return f"ERROR: {e}"


# 测试计算器
test_cases = [
    "15 - 7 + 3",
    "(3 + 4) * 5 / 7",
    "sqrt(144)",
    "2 ** 10",
]

for expr in test_cases:
    print(f"  {expr:25} => {calculator(expr)}")

  15 - 7 + 3                => 11
  (3 + 4) * 5 / 7           => 5
  sqrt(144)                 => 12
  2 ** 10                   => 1024


## 6. 解析模型输出并执行工具调用

In [8]:
def parse_model_output(text: str) -> dict:
    """
    解析模型输出，提取 think / tool_call / answer 三个字段。
    返回 dict，缺失字段为 None。
    """
    result = {"think": None, "tool_call": None, "answer": None}
    
    for tag in ["think", "tool_call", "answer"]:
        pattern = rf"<{tag}>(.*?)</{tag}>"
        match = re.search(pattern, text, re.DOTALL)
        if match:
            result[tag] = match.group(1).strip()
    
    return result


def execute_with_tool(model_output: str) -> dict:
    """
    解析模型输出，执行工具调用，返回完整结果。
    """
    parsed = parse_model_output(model_output)
    
    tool_result = None
    if parsed["tool_call"]:
        tool_result = calculator(parsed["tool_call"])
    
    return {
        **parsed,
        "tool_result": tool_result,
    }


# 用上面模型的输出测试
result = execute_with_tool(response)
print("解析结果:")
for k, v in result.items():
    print(f"  {k:12}: {v}")

解析结果:
  think       : None
  tool_call   : Python 表达式（例如：15 - 7 + 3）
  answer      : 11
  tool_result : ERROR: invalid character '（' (U+FF08) (<string>, line 1)


## 7. 加载 GSM8K 数据集预览

GSM8K 是小学数学题的标准 benchmark，每题有明确的数字答案，适合 RLVR 训练。

In [9]:
import json

GSM8K_DIR = os.path.expanduser("~/.cache/modelscope/datasets/gsm8k/data")

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

gsm8k_train = load_jsonl(f"{GSM8K_DIR}/train.jsonl")
gsm8k_test  = load_jsonl(f"{GSM8K_DIR}/test.jsonl")

# 包装成统一接口，兼容后续代码
class SimpleDataset:
    def __init__(self, data): self._data = data
    def __len__(self): return len(self._data)
    def __getitem__(self, i): return self._data[i]

gsm8k = {"train": SimpleDataset(gsm8k_train), "test": SimpleDataset(gsm8k_test)}
print(f"训练集: {len(gsm8k['train'])} 条")
print(f"测试集: {len(gsm8k['test'])} 条")

训练集: 7473 条
测试集: 1319 条


In [10]:
# 查看一条样本
sample = gsm8k["train"][0]
print("题目:", sample["question"])
print("\n解答:", sample["answer"])

题目: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

解答: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [11]:
def extract_final_answer(gsm8k_answer: str) -> str:
    """
    从 GSM8K 答案字符串中提取最终数字。
    GSM8K 格式：推理过程\n#### 最终答案
    """
    match = re.search(r"####\s*([\d,]+)", gsm8k_answer)
    if match:
        return match.group(1).replace(",", "")
    return None


# 验证提取逻辑
for i in range(5):
    s = gsm8k["train"][i]
    ans = extract_final_answer(s["answer"])
    print(f"[{i}] 答案: {ans:>8}  |  题目片段: {s['question'][:50]}...")

[0] 答案:       72  |  题目片段: Natalia sold clips to 48 of her friends in April, ...
[1] 答案:       10  |  题目片段: Weng earns $12 an hour for babysitting. Yesterday,...
[2] 答案:        5  |  题目片段: Betty is saving money for a new wallet which costs...
[3] 答案:       42  |  题目片段: Julie is reading a 120-page book. Yesterday, she w...
[4] 答案:      624  |  题目片段: James writes a 3-page letter to 2 different friend...


## 8. 端到端流程验证

完整流程：题目 → 模型生成 → 解析工具调用 → 计算结果 → 对比正确答案

In [12]:
import time

def solve_gsm8k(question: str) -> dict:
    """端到端解题：模型 + 工具调用"""
    t0 = time.time()
    
    raw_output = chat([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ], max_new_tokens=512)
    
    elapsed = time.time() - t0
    result = execute_with_tool(raw_output)
    result["raw_output"] = raw_output
    result["elapsed"] = elapsed
    return result


# 测试前 3 道 GSM8K 题目（未经 RLVR 训练，这是 baseline）
print("=== Baseline 测试（未训练）===")
print()

for i in range(3):
    sample = gsm8k["test"][i]
    correct_answer = extract_final_answer(sample["answer"])
    
    result = solve_gsm8k(sample["question"])
    
    model_answer = result["answer"]
    is_correct = (model_answer == correct_answer) if model_answer else False
    
    print(f"[题目 {i+1}] {sample['question'][:60]}...")
    print(f"  正确答案: {correct_answer}")
    print(f"  模型答案: {model_answer}")
    print(f"  工具调用: {result['tool_call']}")
    print(f"  工具结果: {result['tool_result']}")
    print(f"  是否正确: {'✓' if is_correct else '✗'}  ({result['elapsed']:.1f}s)")
    print()

=== Baseline 测试（未训练）===

[题目 1] Janet’s ducks lay 16 eggs per day. She eats three for breakf...
  正确答案: 18
  模型答案: None
  工具调用: None
  工具结果: None
  是否正确: ✗  (14.5s)

[题目 2] A robe takes 2 bolts of blue fiber and half that much white ...
  正确答案: 3
  模型答案: 11
  工具调用: Python 表达式（例如：2 + 3 * 4）
  工具结果: ERROR: invalid character '（' (U+FF08) (<string>, line 1)
  是否正确: ✗  (2.8s)

[题目 3] Josh decides to try flipping a house.  He buys a house for $...
  正确答案: 70000
  模型答案: 18,000
  工具调用: Python 表达式（例如：2 + 3 * 4）
  工具结果: ERROR: invalid character '（' (U+FF08) (<string>, line 1)
  是否正确: ✗  (3.0s)



## 9. 小结

至此，环境验证完成：

| 组件 | 状态 |
|---|---|
| Qwen3.5-2B 模型加载 | ✓ |
| MPS 推理 | ✓ |
| 工具调用格式解析 | ✓ |
| 计算器工具 | ✓ |
| GSM8K 数据集加载 | ✓ |
| 端到端流程 | ✓ |

**观察**：
- 未训练的模型可能不稳定地遵循工具调用格式
- 即使调用了工具，答案也不一定正确
- 这正是 RLVR 要解决的问题：通过 Reward 信号训练模型**稳定地**调用工具并给出正确答案

**下一步**：`02_reward_design.ipynb` — 设计 Reward 函数